<a href="https://colab.research.google.com/github/virusds7778-coder/HSE_Denis_Ivanov_HOMEWORK_2026_2027/blob/main/%D0%9C%D0%9B%D0%98%D0%A2%D0%95251_Ivanov_Denis_Sergeevich_Homework_3_%D0%9A%D0%BB%D1%8E%D1%87%D0%B5%D0%B2%D1%8B%D0%B5_%D1%82%D0%B5%D1%85%D0%BD%D0%BE%D0%BB%D0%BE%D0%B3%D0%B8%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Автоматическое заполнение юридических документов через GigaChat

**Задание 1.** Формирование черновика договора аренды по данным клиента.

**Задание 2.** Извлечение структурированных фактов из текста договора в JSON.

Перед запуском в Google Colab:
1. Нажмите значок ключа слева — **Секреты**.
2. Создайте секрет `GIGACHAT_CREDENTIALS`.
3. Вставьте в него **ключ авторизации** из GigaChat ID.
4. Включите доступ секрета к блокноту.

In [1]:
!pip install -q gigachat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 1.8 MB/s eta 0:00:00


In [20]:
from gigachat import GigaChat
from google.colab import userdata
import json, re

giga = GigaChat(
    credentials=("MDE5ZWNiNTgtMTNhNS03OWNkLTk0YjQtMDY0Nzc0N2ZjM2QxOmE1NzhhNTlmLTVjYjMtNDA3Mi04ZWM0LTBkMmY1MWIzN2Y2OA=="),
    scope="GIGACHAT_API_PERS",
    verify_ssl_certs=False
)

models_response = giga.get_models()

available_models = [
    m.id_ for m in models_response.data
    if getattr(m, "id_", None) and not m.id_.startswith("Embeddings")
]

print("Доступные модели:", available_models)

if not available_models:
    raise RuntimeError("Не удалось получить список моделей GigaChat.")

MODEL_NAME = "GigaChat-2"
print("Используется модель:", MODEL_NAME)

def ask_llm(system: str, user: str, temperature: float = 0.2):
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ],
        "temperature": temperature,
        "max_tokens": 4000
    }

    response = giga.chat(payload)
    return response.choices[0].message.content

print("GigaChat подключен.")

Доступные модели: ['GigaChat-2', 'GigaChat-2-Max', 'GigaChat-2-Pro', 'GigaChat-3-Lightning', 'GigaChat-3-Pro', 'GigaChat-3-Ultra', 'GigaEmbeddings-3B-2025-09']
Используется модель: GigaChat-2
GigaChat подключен.


## Задание 1. Черновик договора аренды

Промпт принимает данные клиента и формирует черновик договора. Если данных недостаточно, модель оставляет маркер `[УТОЧНИТЬ]`, а не придумывает реквизиты.

In [21]:
contract_system_prompt = """
Ты — помощник юриста по российскому гражданскому праву.

Задача: на основе предоставленных данных подготовить ЧЕРНОВИК договора аренды нежилого помещения.

Требования:
1. Используй только данные, переданные пользователем.
2. Если обязательное условие отсутствует, вставь маркер: [УТОЧНИТЬ: что именно нужно указать].
3. Не выдумывай ФИО, ИНН, ОГРН, адреса, банковские реквизиты, кадастровые номера, суммы и даты.
4. Включи разделы:
   - Предмет договора;
   - Права и обязанности арендодателя;
   - Права и обязанности арендатора;
   - Арендная плата и порядок расчетов;
   - Срок аренды;
   - Передача помещения;
   - Ответственность сторон;
   - Изменение и расторжение договора;
   - Форс-мажор;
   - Разрешение споров;
   - Заключительные положения;
   - Реквизиты и подписи сторон.
5. Учитывай, что для договора аренды должны быть определены объект аренды и размер арендной платы.
6. В конце добавь блок «Требуют уточнения» со списком незаполненных существенных условий.
7. Не выдавай договор за юридическую консультацию и не утверждай, что документ готов к подписанию без проверки юристом.

Формат ответа: готовый текст договора на русском языке.
"""

client_data = {
    "тип_договора": "аренда нежилого помещения",
    "арендодатель": {
        "наименование": "ООО «Ромашка»",
        "ИНН": "3123000000",
        "ОГРН": "1123123000000",
        "адрес": "г. Старый Оскол, ул. Ленина, д. 1",
        "представитель": "Генеральный директор Иванов Иван Иванович",
        "основание_полномочий": "Устав"
    },
    "арендатор": {
        "наименование": "ИП Петров Петр Петрович",
        "ИНН": "312300000001",
        "ОГРНИП": "312300000000001",
        "адрес": "г. Старый Оскол, ул. Мира, д. 2"
    },
    "объект_аренды": {
        "адрес": "г. Старый Оскол, ул. Ленина, д. 1, офис 10",
        "площадь_кв_м": 45,
        "кадастровый_номер": "[УТОЧНИТЬ]"
    },
    "арендная_плата": {
        "размер_руб_в_месяц": 30000,
        "ндс": "не облагается, арендодатель применяет УСН",
        "срок_оплаты": "до 5 числа текущего месяца"
    },
    "срок_аренды": "11 месяцев с 01.11.2026 по 30.09.2027",
    "обеспечительный_платеж": 30000,
    "коммунальные_платежи": "оплачивает арендатор по показаниям счетчиков"
}

contract_user_prompt = f"""
Сформируй черновик договора аренды на русском языке по следующим данным клиента:

{json.dumps(client_data, ensure_ascii=False, indent=2)}
"""

contract_draft = ask_llm(contract_system_prompt, contract_user_prompt)
print(contract_draft)

**Черновик договора аренды нежилого помещения**

Настоящий договор аренды нежилого помещения заключен между:
- АРЕНДОДАТЕЛЕМ: ООО «Ромашка», ИНН 3123000000, ОГРН 1123123000000, адрес: г. Старый Оскол, ул. Ленина, д. 1; представитель – Генеральный директор Иванов Иван Иванович, действующий на основании Устава.
- АРЕНДАТОРОМ: ИП Петров Петр Петрович, ИНН 312300000001, ОГРНИП 312300000000001, адрес: г. Старый Оскол, ул. Мира, д. 2.

**Предмет договора**

1. Арендатором арендуется нежилое помещение общей площадью 45 кв.м, расположенное по адресу: г. Старый Оскол, ул. Ленина, д. 1, офис 10, с кадастровым номером [УТОЧНИТЬ]. Помещение пригодно для использования под административные цели.

**Права и обязанности арендодателя**

Арендодатель обязуется:
- Предоставить помещение в состоянии, пригодном для целей аренды, свободным от прав третьих лиц.
- Обеспечить доступ к инженерным коммуникациям и поддерживать их исправность.
- Устранять неисправности коммуникаций, находящихся в помещении, в тече

In [22]:
with open("договор_аренды_черновик.txt", "w", encoding="utf-8") as f:
    f.write(contract_draft)

from google.colab import files
files.download("договор_аренды_черновик.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Задание 2. Извлечение фактов в JSON

Модель извлекает стороны договора, объект аренды, срок, размер арендной платы и другие условия. Ответ должен быть строго валидным JSON без пояснений.

In [23]:
extraction_system_prompt = """
Ты — система извлечения структурированных данных из юридических документов.

Задача: извлеки факты из текста договора аренды и верни ТОЛЬКО JSON-объект.

Правила:
1. Не добавляй пояснения, markdown, комментарии или текст до/после JSON.
2. Если значение отсутствует в тексте, используй null.
3. Не выдумывай данные.
4. Даты приводи в формате ГГГГ-ММ-ДД.
5. Суммы указывай числами без пробелов и символа рубля.
6. Верни JSON строго по схеме:

{
  "договор": {
    "тип": "строка",
    "номер": "строка или null",
    "дата_заключения": "строка или null",
    "место_заключения": "строка или null"
  },
  "арендодатель": {
    "наименование": "строка или null",
    "инн": "строка или null",
    "огрн": "строка или null",
    "адрес": "строка или null",
    "представитель": "строка или null"
  },
  "арендатор": {
    "наименование": "строка или null",
    "инн": "строка или null",
    "огрн": "строка или null",
    "адрес": "строка или null",
    "представитель": "строка или null"
  },
  "объект_аренды": {
    "описание": "строка или null",
    "адрес": "строка или null",
    "площадь_кв_м": "число или null",
    "кадастровый_номер": "строка или null"
  },
  "условия": {
    "срок_начало": "строка или null",
    "срок_окончание": "строка или null",
    "арендная_плата_руб_в_месяц": "число или null",
    "ндс": "строка или null",
    "порядок_оплаты": "строка или null",
    "обеспечительный_платеж_руб": "число или null",
    "коммунальные_платежи": "строка или null"
  }
}
"""

contract_text = """
ДОГОВОР АРЕНДЫ НЕЖИЛОГО ПОМЕЩЕНИЯ № 17-А
г. Старый Оскол                                                                                   01 ноября 2026 г.

ООО «Ромашка», ИНН 3123000000, ОГРН 1123123000000, в лице генерального директора Иванова Ивана Ивановича,
действующего на основании Устава, именуемое «Арендодатель», и
ИП Петров Петр Петрович, ИНН 312300000001, ОГРНИП 312300000000001,
именуемый «Арендатор», заключили настоящий договор о следующем.

1. Арендодатель передает Арендатору за плату во временное владение и пользование нежилое помещение,
расположенное по адресу: г. Старый Оскол, ул. Ленина, д. 1, офис 10, площадью 45 кв. м.
Кадастровый номер помещения: 31:23:0000000:1234.

2. Срок аренды: с 01.11.2026 по 30.09.2027.

3. Арендная плата составляет 30 000 рублей в месяц, НДС не облагается.
Плата вносится до 5 числа текущего месяца.

4. Арендатор перечисляет обеспечительный платеж в размере 30 000 рублей.

5. Коммунальные платежи оплачивает Арендатор по показаниям приборов учета.
"""

extraction_user_prompt = f"""
Извлеки данные из следующего текста договора и верни только JSON:

{contract_text}
"""

raw_json = ask_llm(extraction_system_prompt, extraction_user_prompt, temperature=0)

match = re.search(r"\{.*\}", raw_json, re.DOTALL)
extracted = json.loads(match.group(0))

print(json.dumps(extracted, ensure_ascii=False, indent=2))

{
  "договор": {
    "тип": "АРЕНДА",
    "номер": "17-А",
    "дата_заключения": "01 ноября 2026 г.",
    "место_заключения": "г. Старый Оскол"
  },
  "арендодатель": {
    "наименование": "ООО «Ромашка»",
    "инн": "3123000000",
    "огрн": "1123123000000",
    "адрес": null,
    "представитель": "Иванов Иван Иванович"
  },
  "арендатор": {
    "наименование": "ИП Петров Петр Петрович",
    "инн": "312300000001",
    "огрн": "312300000000001",
    "адрес": null,
    "представитель": null
  },
  "объект_аренды": {
    "описание": "нежилое помещение",
    "адрес": "г. Старый Оскол, ул. Ленина, д. 1, офис 10",
    "площадь_кв_м": 45,
    "кадастровый_номер": "31:23:0000000:1234"
  },
  "условия": {
    "срок_начало": "01.11.2026",
    "срок_окончание": "30.09.2027",
    "арендная_плата_руб_в_месяц": 30000,
    "ндс": "не облагается",
    "порядок_оплаты": "до 5 числа текущего месяца",
    "обеспечительный_платеж_руб": 30000,
    "коммунальные_платежи": "оплачивает Арендатор по показани

In [24]:
with open("извлеченные_факты.json", "w", encoding="utf-8") as f:
    json.dump(extracted, f, ensure_ascii=False, indent=2)

from google.colab import files
files.download("извлеченные_факты.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Вывод

GigaChat:
1. сформировал черновик договора аренды на основе данных клиента;
2. обозначил недостающие сведения маркером `[УТОЧНИТЬ]`;
3. извлек ключевые факты из текста договора в структурированный JSON.

Сгенерированный договор является черновиком и требует проверки юристом перед использованием.